<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P04/Pratica4_Visualizacao_Integrada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parte 6 — Visualização integrada
Lê as saídas do notebook ProtT5 (`ss3_preds.fasta`, `disorder_preds.fasta`), do TMbed (`predictions.txt`) e do CD-search (`hitdata.txt`) e desenha os quatro tracks alinhados por proteína, sem precisar copiar nada manualmente.

**Como usar:** rode as Partes 2, 3 e 5 normalmente primeiro. No CD-search, baixe os dados completos pela opção "Download complete search data" (formato tab-delimited / hitdata.txt) na página de resultados. Depois suba os arquivos que tiver (pode rodar só com alguns — os tracks que faltarem simplesmente não aparecem) quando pedido abaixo.

In [ ]:
#@title Enviar os arquivos de saída { display-mode: "form" }
from google.colab import files

print('Selecione os arquivos disponíveis: ss3_preds.fasta, disorder_preds.fasta, '
      'predictions.txt, hitdata.txt (pode selecionar só alguns)')
uploaded = files.upload()


Selecione os arquivos disponíveis: ss3_preds.fasta, disorder_preds.fasta, predictions.txt, hitdata.txt (pode selecionar só alguns)


Saving hitdata.txt to hitdata.txt


In [ ]:
#@title Identificar cada arquivo pelo conteúdo { display-mode: "form" }
# Em vez de confiar no nome do arquivo (o aluno pode ter renomeado ao baixar),
# identifica cada um pela primeira linha característica de cada saída.
ss3_text, diso_text, tm_text, cd_text = '', '', '', ''

for fname, content in uploaded.items():
    text = content.decode('utf-8', errors='replace')
    stripped = text.lstrip()
    if stripped.startswith('#Batch CD-search') or '\tHit type\t' in text:
        cd_text = text
        print(f'{fname} -> reconhecido como CD-search (hitdata.txt)')
        continue
    first_non_header = ''
    for line in text.splitlines():
        if line.strip() and not line.startswith('>'):
            first_non_header = line.strip()
            break
    sample = set(first_non_header[:200])
    if sample <= set('HEL'):
        ss3_text = text
        print(f'{fname} -> reconhecido como estrutura secundária (SS3)')
    elif sample <= set('OD'):
        diso_text = text
        print(f'{fname} -> reconhecido como desordem')
    elif sample <= set('SHhBb.'):
        tm_text = text
        print(f'{fname} -> reconhecido como TMbed')
    else:
        print(f'{fname} -> NÃO reconhecido automaticamente (conteúdo: "{first_non_header[:40]}..."). '
              f'Confira se é um dos formatos esperados.')

if not any([ss3_text, diso_text, tm_text, cd_text]):
    raise ValueError('Nenhum arquivo foi reconhecido. Rode a célula de novo e reenvie os arquivos.')


hitdata.txt -> reconhecido como CD-search (hitdata.txt)


In [ ]:
#@title Renderizar a visualização { display-mode: "form" }
from IPython.display import HTML
import json

_html_template = '<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="UTF-8">\n<title>Prática 4 — Visualizador de Predições</title>\n<link rel="preconnect" href="https://fonts.googleapis.com">\n<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>\n<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500;600&display=swap" rel="stylesheet">\n<style>\n  :root{\n    --bg: #0F1620;\n    --panel: #16202C;\n    --panel-2: #1B2733;\n    --grid: #263442;\n    --text: #E8EDF2;\n    --muted: #7E90A3;\n    --muted-2: #56687A;\n\n    --helix: #D9694F;\n    --strand: #4FA9A2;\n    --loop: #2B3947;\n\n    --disorder: #9B8CE0;\n    --order: #223040;\n\n    --tm-in: #D98F3B;\n    --tm-out: #8C5A25;\n    --signal: #E8C171;\n    --tm-none: #223040;\n\n    --dom-specific: #5FB88F;\n    --dom-multi: #C77DBB;\n    --dom-other: #3D4E5E;\n\n    --radius: 3px;\n  }\n  *{ box-sizing: border-box; }\n  body{\n    margin:0;\n    background: var(--bg);\n    color: var(--text);\n    font-family: \'Space Grotesk\', sans-serif;\n    padding: 40px 24px 80px;\n  }\n  .wrap{ max-width: 980px; margin: 0 auto; }\n\n  header{ margin-bottom: 36px; }\n  header .kicker{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 12.5px;\n    color: var(--muted);\n    letter-spacing: 0.02em;\n    margin-bottom: 8px;\n  }\n  h1{\n    font-size: 28px;\n    font-weight: 600;\n    margin: 0 0 10px;\n    letter-spacing: -0.01em;\n  }\n  header p{\n    color: var(--muted);\n    font-size: 14.5px;\n    line-height: 1.6;\n    max-width: 640px;\n    margin: 0;\n  }\n\n  .inputs{\n    display: grid;\n    grid-template-columns: 1fr;\n    gap: 14px;\n    margin-bottom: 20px;\n  }\n  .input-block{\n    background: var(--panel);\n    border: 1px solid var(--grid);\n    border-radius: var(--radius);\n    padding: 14px 16px 16px;\n  }\n  .input-block label{\n    display:block;\n    font-size: 13px;\n    font-weight: 600;\n    margin-bottom: 3px;\n  }\n  .input-block .hint{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 11.5px;\n    color: var(--muted-2);\n    margin-bottom: 8px;\n  }\n  textarea{\n    width: 100%;\n    min-height: 90px;\n    background: #0B121A;\n    border: 1px solid var(--grid);\n    border-radius: var(--radius);\n    color: var(--text);\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 12px;\n    padding: 10px 12px;\n    resize: vertical;\n    line-height: 1.5;\n  }\n  textarea::placeholder{ color: var(--muted-2); }\n  textarea:focus, button:focus-visible{\n    outline: 2px solid var(--tm-in);\n    outline-offset: 1px;\n  }\n\n  .actions{\n    display:flex;\n    align-items:center;\n    gap: 14px;\n    margin-bottom: 34px;\n  }\n  button{\n    font-family: \'Space Grotesk\', sans-serif;\n    font-size: 14px;\n    font-weight: 600;\n    padding: 10px 20px;\n    border-radius: var(--radius);\n    border: 1px solid transparent;\n    cursor: pointer;\n  }\n  #renderBtn{ background: var(--tm-in); color: #14100A; border-color: var(--tm-in); }\n  #renderBtn:hover{ background: #E8A05C; }\n  #clearBtn{ background: transparent; color: var(--muted); border-color: var(--grid); }\n  #clearBtn:hover{ color: var(--text); border-color: var(--muted-2); }\n  .status{ font-family: \'IBM Plex Mono\', monospace; font-size: 12px; color: var(--muted); }\n\n  .legend{\n    display:flex;\n    flex-wrap: wrap;\n    gap: 18px;\n    padding: 14px 16px;\n    background: var(--panel-2);\n    border: 1px solid var(--grid);\n    border-radius: var(--radius);\n    margin-bottom: 32px;\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 11.5px;\n    color: var(--muted);\n  }\n  .legend .grp{ display:flex; align-items:center; gap: 8px; }\n  .legend .grp .items{ display:flex; gap: 10px; }\n  .legend .chip{ display:flex; align-items:center; gap:5px; }\n  .swatch{ width: 11px; height: 11px; border-radius: 2px; display:inline-block; }\n\n  .protein{\n    border: 1px solid var(--grid);\n    border-radius: var(--radius);\n    margin-bottom: 22px;\n    overflow: hidden;\n  }\n  .protein-head{\n    display:flex;\n    align-items: baseline;\n    justify-content: space-between;\n    flex-wrap: wrap;\n    gap: 10px 24px;\n    padding: 16px 18px;\n    background: var(--panel);\n    border-bottom: 1px solid var(--grid);\n  }\n  .protein-head .name{ font-size: 17px; font-weight: 600; }\n  .protein-head .len{ font-family: \'IBM Plex Mono\', monospace; color: var(--muted); font-size: 12.5px; }\n  .stats{\n    display:flex;\n    gap: 18px;\n    flex-wrap: wrap;\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 12px;\n  }\n  .stats .stat b{ color: var(--text); font-weight: 600; }\n  .stats .stat{ color: var(--muted); }\n\n  .tracks{ padding: 18px; background: var(--panel-2); }\n  .track-row{ margin-bottom: 16px; }\n  .track-row:last-child{ margin-bottom: 0; }\n  .seq-line{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 12px;\n    line-height: 1;\n    display: flex;\n    white-space: nowrap;\n    margin-bottom: 3px;\n  }\n  .seq-line span{ width: 8.4px; text-align:center; color: var(--text); }\n  .track-strip{\n    display:flex;\n    height: 9px;\n    margin-bottom: 2px;\n  }\n  .track-strip span{ width: 8.4px; height: 100%; }\n  .track-label{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 10px;\n    color: var(--muted-2);\n    margin-bottom: 3px;\n    margin-top: 8px;\n  }\n  .pos-ruler{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 9.5px;\n    color: var(--muted-2);\n    margin-bottom: 4px;\n  }\n\n  .domain-lanes-inline{\n    position: relative;\n    margin-bottom: 2px;\n  }\n  .domain-box-inline{\n    position: absolute;\n    top: 0;\n    height: 12px;\n    border-radius: 2px;\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 8.5px;\n    line-height: 12px;\n    color: #0B121A;\n    padding-left: 3px;\n    overflow: hidden;\n    white-space: nowrap;\n    cursor: default;\n    box-sizing: border-box;\n  }\n  .domain-note{\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 11px;\n    color: var(--muted-2);\n    margin-top: 4px;\n  }\n\n  .empty{\n    padding: 40px 20px;\n    text-align: center;\n    color: var(--muted-2);\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 13px;\n    border: 1px dashed var(--grid);\n    border-radius: var(--radius);\n  }\n</style>\n</head>\n<body>\n<div class="wrap">\n\n  <header>\n    <div class="kicker">PRÁTICA 4 — BIOMOLCOMP</div>\n    <h1>Visualizador de predições por resíduo</h1>\n    <p>Cole abaixo as saídas brutas do ProtT5, do TMbed e do CD-search. A ferramenta alinha os quatro tracks pelo identificador da proteína: estrutura secundária, desordem e regiões transmembrana resíduo por resíduo, e os domínios do CD-search como blocos posicionados por coordenada.</p>\n  </header>\n\n  <div class="inputs">\n    <div class="input-block">\n      <label for="ss3in">Estrutura secundária — ss3_preds.fasta</label>\n      <div class="hint">formato: &gt;ID seguido da string H/E/L (saída do notebook PredictSS_embed_ProtT5)</div>\n      <textarea id="ss3in" placeholder="&gt;Protein1|Chain |Prot1|Homo sapiens (9606)&#10;LLLLLLHHHHHLLL..."></textarea>\n    </div>\n    <div class="input-block">\n      <label for="disoin">Desordem — disorder_preds.fasta</label>\n      <div class="hint">formato: &gt;ID seguido da string O/D</div>\n      <textarea id="disoin" placeholder="&gt;Protein1|Chain |Prot1|Homo sapiens (9606)&#10;DDDDDDOOOOOO..."></textarea>\n    </div>\n    <div class="input-block">\n      <label for="tmin">TMbed — predictions.txt (--out-format 0)</label>\n      <div class="hint">formato de 3 linhas: &gt;ID / sequência / string S·H·h·B·b·.</div>\n      <textarea id="tmin" placeholder="&gt;Protein1|Chain |Prot1|Homo sapiens (9606)&#10;MSYPGYPP...&#10;............"></textarea>\n    </div>\n    <div class="input-block">\n      <label for="cdin">Domínios — CD-search (hitdata.txt)</label>\n      <div class="hint">na página de resultados do CD-search: "Download complete search data" → formato tab-delimited (hitdata.txt)</div>\n      <textarea id="cdin" placeholder="#Batch CD-search tool NIH/NLM/NCBI&#10;#cdsid ...&#10;Query&#9;Hit type&#9;PSSM-ID&#9;From&#9;To&#9;E-Value&#9;Bitscore&#9;Accession&#9;Short name&#9;Incomplete&#9;Superfamily&#10;Q#1&#9;-&gt;Protein1|...&#9;specific&#9;...&#9;..."></textarea>\n    </div>\n  </div>\n\n  <div class="actions">\n    <button id="renderBtn">Visualizar</button>\n    <button id="clearBtn">Limpar</button>\n    <span class="status" id="status"></span>\n  </div>\n\n  <div class="legend">\n    <div class="grp">\n      <span>estrutura</span>\n      <div class="items">\n        <span class="chip"><span class="swatch" style="background:var(--helix)"></span>hélice (H)</span>\n        <span class="chip"><span class="swatch" style="background:var(--strand)"></span>fita (E)</span>\n        <span class="chip"><span class="swatch" style="background:var(--loop)"></span>loop (L)</span>\n      </div>\n    </div>\n    <div class="grp">\n      <span>desordem</span>\n      <div class="items">\n        <span class="chip"><span class="swatch" style="background:var(--disorder)"></span>desordenado (D)</span>\n        <span class="chip"><span class="swatch" style="background:var(--order)"></span>ordenado (O)</span>\n      </div>\n    </div>\n    <div class="grp">\n      <span>membrana</span>\n      <div class="items">\n        <span class="chip"><span class="swatch" style="background:var(--signal)"></span>sinal (S)</span>\n        <span class="chip"><span class="swatch" style="background:var(--tm-in)"></span>TM dentro→fora (H)</span>\n        <span class="chip"><span class="swatch" style="background:var(--tm-out)"></span>TM fora→dentro (h)</span>\n      </div>\n    </div>\n    <div class="grp">\n      <span>domínios</span>\n      <div class="items">\n        <span class="chip"><span class="swatch" style="background:var(--dom-specific)"></span>hit específico</span>\n        <span class="chip"><span class="swatch" style="background:var(--dom-multi)"></span>multi-domínio</span>\n      </div>\n    </div>\n  </div>\n\n  <div id="output"><div class="empty">Cole os dados acima e clique em Visualizar.</div></div>\n\n</div>\n\n<script>\nfunction parseFasta(text){\n  const records = {};\n  const lines = text.split(/\\r?\\n/);\n  let curId = null, buf = [];\n  const flush = () => { if(curId) records[curId] = (records[curId]||\'\') + buf.join(\'\'); buf=[]; };\n  for(const raw of lines){\n    const line = raw.trim();\n    if(!line) continue;\n    if(line.startsWith(\'>\')){\n      flush();\n      curId = line.slice(1).split(\'|\')[0].trim().split(/\\s+/)[0];\n      if(!curId) curId = line.slice(1).trim();\n    } else {\n      buf.push(line.replace(/\\s/g,\'\'));\n    }\n  }\n  flush();\n  return records;\n}\n\n// TMbed 3-line format: header, sequence, track. Also tolerate the fasta-style\n// 2-line variant some out-formats produce.\nfunction parseTmbed(text){\n  const seqs = {}, tracks = {};\n  const lines = text.split(/\\r?\\n/).map(l=>l.trim()).filter(l=>l.length);\n  let i = 0;\n  while(i < lines.length){\n    if(lines[i].startsWith(\'>\')){\n      const id = lines[i].slice(1).split(\'|\')[0].trim().split(/\\s+/)[0];\n      const seq = lines[i+1] || \'\';\n      const track = lines[i+2] || \'\';\n      seqs[id] = seq;\n      tracks[id] = track;\n      i += 3;\n    } else { i++; }\n  }\n  return {seqs, tracks};\n}\n\n// CD-search "Download complete search data" (hitdata.txt) tab-delimited format.\n// Comment lines start with \'#\'; the header row starts with \'Query\'; each data\n// row\'s first column looks like "Q#1 - >Protein1|Chain |...".\nfunction parseCdsearch(text){\n  const hits = {};\n  let headerSeen = false;\n  for(const raw of text.split(/\\r?\\n/)){\n    const line = raw.replace(/\\r$/, \'\');\n    if(!line.trim() || line.startsWith(\'#\')) continue;\n    const cols = line.split(\'\\t\');\n    if(cols[0] === \'Query\'){ headerSeen = true; continue; }\n    if(!headerSeen || cols.length < 11) continue;\n    const [queryRaw, hitType, pssmId, from, to, evalue, bitscore, accession, shortName, incomplete, superfamily] = cols;\n    const m = queryRaw.match(/>([^\\t]+)/);\n    if(!m) continue;\n    const id = m[1].split(\'|\')[0].trim().split(/\\s+/)[0];\n    if(!hits[id]) hits[id] = [];\n    hits[id].push({\n      hitType: hitType.trim(),\n      pssmId: pssmId.trim(),\n      from: parseInt(from, 10),\n      to: parseInt(to, 10),\n      evalue: evalue.trim(),\n      bitscore: parseFloat(bitscore),\n      accession: accession.trim(),\n      shortName: shortName.trim(),\n      incomplete: incomplete.trim(),\n      superfamily: superfamily.trim(),\n    });\n  }\n  return hits;\n}\n\n// Greedy interval scheduling: stack overlapping domain hits into the fewest\n// horizontal lanes so nothing visually overlaps.\nfunction assignLanes(hits){\n  const sorted = [...hits].sort((a,b) => a.from - b.from);\n  const laneEnds = [];\n  const placed = [];\n  for(const h of sorted){\n    let lane = laneEnds.findIndex(end => end < h.from);\n    if(lane === -1){ lane = laneEnds.length; laneEnds.push(h.to); }\n    else { laneEnds[lane] = h.to; }\n    placed.push({...h, lane});\n  }\n  return {items: placed, laneCount: laneEnds.length || 1};\n}\n\nfunction domainColor(hitType){\n  if(hitType === \'specific\') return \'var(--dom-specific)\';\n  if(hitType === \'multi-dom\') return \'var(--dom-multi)\';\n  return \'var(--dom-other)\';\n}\n\n// Draws the slice of each domain hit that falls within [chunkStart, chunkStart+chunkLen)\n// (0-indexed, half-open) using the same 8.4px-per-residue grid as the sequence and\n// the other track strips, so domain boxes line up column-for-column with SS3/disorder/TM.\n// `items` are pre-laned (see assignLanes) so a hit keeps the same lane across every line.\nconst CELL_W = 8.4;\nconst LANE_H = 13;\n\nfunction domainChunkHtml(items, laneCount, chunkStart, chunkLen){\n  let boxes = \'\';\n  for(const h of items){\n    const hFrom0 = h.from - 1, hTo0 = h.to - 1; // to 0-indexed\n    const overlapStart = Math.max(hFrom0, chunkStart);\n    const overlapEnd = Math.min(hTo0, chunkStart + chunkLen - 1);\n    if(overlapStart > overlapEnd) continue;\n    const left = (overlapStart - chunkStart) * CELL_W;\n    const width = (overlapEnd - overlapStart + 1) * CELL_W;\n    const top = h.lane * LANE_H;\n    const label = overlapStart === hFrom0 ? (h.shortName || h.accession) : \'\';\n    const title = `${h.shortName} (${h.accession}) \\u00b7 ${h.from}-${h.to} \\u00b7 E-value ${h.evalue}`;\n    boxes += `<div class="domain-box-inline" title="${title}" style="left:${left}px;width:${width}px;top:${top}px;background:${domainColor(h.hitType)}">${label}</div>`;\n  }\n  return `<div class="domain-lanes-inline" style="height:${laneCount*LANE_H}px">${boxes}</div>`;\n}\n\nfunction countChar(str, ch){ let n=0; for(const c of str) if(c===ch) n++; return n; }\n\nfunction countRuns(str, chars){\n  let runs = 0, inRun = false;\n  for(const c of str){\n    const hit = chars.includes(c);\n    if(hit && !inRun) runs++;\n    inRun = hit;\n  }\n  return runs;\n}\n\nfunction classFor(kind, ch){\n  if(kind===\'ss3\'){\n    if(ch===\'H\') return \'helix\';\n    if(ch===\'E\') return \'strand\';\n    return \'loop\';\n  }\n  if(kind===\'diso\'){\n    return ch===\'D\' ? \'disorder\' : \'order\';\n  }\n  if(kind===\'tm\'){\n    if(ch===\'S\') return \'signal\';\n    if(ch===\'H\') return \'tm-in\';\n    if(ch===\'h\') return \'tm-out\';\n    return \'tm-none\';\n  }\n}\n\nconst colorVar = {\n  helix:\'--helix\', strand:\'--strand\', loop:\'--loop\',\n  disorder:\'--disorder\', order:\'--order\',\n  signal:\'--signal\', \'tm-in\':\'--tm-in\', \'tm-out\':\'--tm-out\', \'tm-none\':\'--tm-none\'\n};\n\nfunction stripHtml(kind, track, len){\n  let html = \'<div class="track-strip">\';\n  for(let i=0;i<len;i++){\n    const ch = track[i] || \'.\';\n    const cls = classFor(kind, ch);\n    html += `<span style="background:var(${colorVar[cls]})"></span>`;\n  }\n  html += \'</div>\';\n  return html;\n}\n\nfunction seqHtml(seq, len){\n  let html = \'<div class="seq-line">\';\n  for(let i=0;i<len;i++){\n    html += `<span>${seq[i] || \'·\'}</span>`;\n  }\n  html += \'</div>\';\n  return html;\n}\n\nfunction render(){\n  const ss3 = parseFasta(document.getElementById(\'ss3in\').value);\n  const diso = parseFasta(document.getElementById(\'disoin\').value);\n  const tmRaw = parseTmbed(document.getElementById(\'tmin\').value);\n  const cdHits = parseCdsearch(document.getElementById(\'cdin\').value);\n\n  const ids = new Set([...Object.keys(ss3), ...Object.keys(diso), ...Object.keys(tmRaw.seqs), ...Object.keys(cdHits)]);\n  const out = document.getElementById(\'output\');\n  const status = document.getElementById(\'status\');\n\n  if(ids.size === 0){\n    out.innerHTML = \'<div class="empty">Nenhum registro reconhecido. Confira o formato dos dados colados.</div>\';\n    status.textContent = \'\';\n    return;\n  }\n\n  let html = \'\';\n  const WRAP = 60;\n\n  for(const id of ids){\n    const seq = tmRaw.seqs[id] || \'\';\n    const hits = cdHits[id] || [];\n    const maxHitTo = hits.length ? Math.max(...hits.map(h => h.to)) : 0;\n    const len = seq.length || Math.max((ss3[id]||\'\').length, (diso[id]||\'\').length, maxHitTo);\n    const ss3Str = ss3[id] || \'\';\n    const disoStr = diso[id] || \'\';\n    const tmStr = tmRaw.tracks[id] || \'\';\n\n    const hPct = ss3Str ? (100*countChar(ss3Str,\'H\')/ss3Str.length).toFixed(0) : \'—\';\n    const ePct = ss3Str ? (100*countChar(ss3Str,\'E\')/ss3Str.length).toFixed(0) : \'—\';\n    const dPct = disoStr ? (100*countChar(disoStr,\'D\')/disoStr.length).toFixed(0) : \'—\';\n    const tmSegs = tmStr ? countRuns(tmStr, [\'H\',\'h\']) : \'—\';\n    const hasSignal = tmStr ? (tmStr.includes(\'S\') ? \'sim\' : \'não\') : \'—\';\n    const relevantHits = hits.filter(h => h.hitType===\'specific\' || h.hitType===\'multi-dom\');\n    const {items: laneItems, laneCount} = assignLanes(relevantHits);\n    const domCount = relevantHits.length;\n\n    html += `<div class="protein">\n      <div class="protein-head">\n        <div>\n          <span class="name">${id}</span>\n          <span class="len"> · ${len || \'?\'} aa</span>\n        </div>\n        <div class="stats">\n          <span class="stat">hélice <b>${hPct}%</b></span>\n          <span class="stat">fita <b>${ePct}%</b></span>\n          <span class="stat">desordem <b>${dPct}%</b></span>\n          <span class="stat">segmentos TM <b>${tmSegs}</b></span>\n          <span class="stat">peptídeo sinal <b>${hasSignal}</b></span>\n          <span class="stat">domínios <b>${hits.length ? domCount : \'—\'}</b></span>\n        </div>\n      </div>\n      <div class="tracks">`;\n\n    if(hits.length && laneItems.length === 0){\n      html += `<div class="domain-note">Domínios encontrados apenas como hit não-específico/superfamily (sem hit "specific" ou "multi-dom") — não desenhados na faixa abaixo.</div>`;\n    }\n\n    for(let start=0; start<len; start+=WRAP){\n      const chunkLen = Math.min(WRAP, len-start);\n      const seqChunk = seq.slice(start, start+chunkLen);\n      const ss3Chunk = ss3Str.slice(start, start+chunkLen);\n      const disoChunk = disoStr.slice(start, start+chunkLen);\n      const tmChunk = tmStr.slice(start, start+chunkLen);\n\n      html += `<div class="track-row">\n        <div class="pos-ruler">${start+1}</div>\n        ${seqHtml(seqChunk, chunkLen)}\n        ${ss3Chunk ? stripHtml(\'ss3\', ss3Chunk, chunkLen) : \'\'}\n        ${disoChunk ? stripHtml(\'diso\', disoChunk, chunkLen) : \'\'}\n        ${tmChunk ? stripHtml(\'tm\', tmChunk, chunkLen) : \'\'}\n        ${laneItems.length ? domainChunkHtml(laneItems, laneCount, start, chunkLen) : \'\'}\n      </div>`;\n    }\n\n    html += `</div></div>`;\n  }\n\n  out.innerHTML = html;\n  status.textContent = `${ids.size} proteína(s) renderizada(s)`;\n}\n\ndocument.getElementById(\'renderBtn\').addEventListener(\'click\', render);\ndocument.getElementById(\'clearBtn\').addEventListener(\'click\', () => {\n  document.getElementById(\'ss3in\').value = \'\';\n  document.getElementById(\'disoin\').value = \'\';\n  document.getElementById(\'tmin\').value = \'\';\n  document.getElementById(\'cdin\').value = \'\';\n  document.getElementById(\'output\').innerHTML = \'<div class="empty">Cole os dados acima e clique em Visualizar.</div>\';\n  document.getElementById(\'status\').textContent = \'\';\n});\n</script>\n\n<script>\nwindow.addEventListener(\'DOMContentLoaded\', function(){\n  document.getElementById(\'ss3in\').value = SS3_TEXT;\n  document.getElementById(\'disoin\').value = DISO_TEXT;\n  document.getElementById(\'tmin\').value = TM_TEXT;\n  document.getElementById(\'cdin\').value = CD_TEXT;\n  document.getElementById(\'renderBtn\').click();\n});\n</script>\n</body>\n</html>\n'

_html = (_html_template
    .replace('SS3_TEXT', json.dumps(ss3_text))
    .replace('DISO_TEXT', json.dumps(diso_text))
    .replace('TM_TEXT', json.dumps(tm_text))
    .replace('CD_TEXT', json.dumps(cd_text)))

display(HTML(_html))
